# RappiPlus Analytics: Converting Data into Strategic Business Decisions

## Introduction

The primary objective of this project is to evaluate the commercial performance of the **RappiPlus** service to enable **data-driven business strategies**. 

To achieve this, I integrated and analyzed multiple business datasets across different environments:

* **rappiplus_orders_raw.csv** -> Transactional data including orders, pricing structures, discounts, and gross revenue.
* **rappiplus_catalog.csv** -> Cost of goods sold (COGS), product categorization, and supplier data.
* **rappiplus_marketing_spend.csv** -> Marketing investment metrics segmented by acquisition channel and country.
* **events / users / user_activity (SQL Database)** -> In-app user behavior logs, clickstream data, and activity patterns.
* **experiment_checkout_ui.csv** -> Granular results from an A/B test conducted on the checkout user interface.

---

## Analytical Workflow & Methodology

The project is structured around a rigorous, end-to-end analytical pipeline designed to address core business challenges:

1. 🔍 **Data Quality & Integrity (Python):** Auditing raw datasets to identify missing values, duplicates, and anomalies, ensuring a reliable foundation for analysis.
2. 💰 **Financial Performance & Profitability:** Analyzing revenue streams, operational costs, and profit margins to isolate financial inefficiencies.
3. 🛒 **Conversion Funnel Analysis (SQL):** Mapping the user journey step-by-step to calculate step-conversion rates and pinpoint exact drop-off stages.
4. 🔁 **User Retention & Cohort Analysis:** Tracking user behavior over time to measure platform loyalty and long-term retention trends.
5. 🧪 **A/B Testing & Statistical Validation:** Running statistical hypothesis testing (Z-test) to validate whether UI changes significantly impact conversion rates.
6. 📊 **Business Intelligence Dashboarding:** Designing interactive dashboards to synthesize technical findings into scalable visual solutions for stakeholders.

Throughout this project, I focused on translating complex datasets into clear, actionable insights and **strategic recommendations** to solve real-world business puzzles.

---

## 🔹 Step 1: Data Ingestion & Quality Assurance

---

### 1.1 Data Ingestion and Initial Exploration

* Load the source files into the environment:
  * `rappiplus_orders_raw.csv`
  * `rappiplus_catalog.csv`
  * `rappiplus_marketing_spend.csv`
* Store the resulting DataFrames as:
  * `orders`, `catalog`, and `marketing`
* Perform an initial exploratory analysis on each dataset to understand their structure.

---

In [1]:
# Import libraries
import pandas as pd

In [2]:
# Load datasets
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [3]:
# explore orders
orders.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [4]:
# explore catalog
catalog.head()

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [5]:
# explorare marketing
marketing.head()

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


---

### Data Quality Review & Cleaning

**🎯 Objective:** Identify, audit, and correct data integrity issues that could misrepresent or skew revenue, cost, and profitability metrics.

A comprehensive quality check is performed across all three datasets:
* **Datetime Standardization:** Validate and parse date features into the correct datetime formats.
* **Numerical Bounds Audit:** Inspect numerical variables to detect anomalies, such as unexpected negative values or invalid zeros.
* **Financial Consistency Checks:** Verify that monetary amounts and transactions align logically across records.
* **Deduplication:** Identify and isolate duplicate entries to maintain data uniqueness.
* **Categorical Feature Validation:** Audit categorical variables for spelling consistency, missing labels, or structural anomalies.

---

#### Data Quality Review: `orders` Dataset

In [6]:
# Inspecting dataframe schema and metadata
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  str    
 1   id_usuario          25100 non-null  str    
 2   fecha_hora_pedido   25100 non-null  str    
 3   pais                24800 non-null  str    
 4   dispositivo         25080 non-null  str    
 5   fuente_referencia   25070 non-null  str    
 6   nombre_producto     25070 non-null  str    
 7   categoria_producto  25020 non-null  str    
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), str(8)
memory usage: 2.3 MB


**✍️ Analytical Notes:** An initial audit reveals missing values across several features, alongside data type inconsistencies. To ensure data integrity without skewing subsequent statistical analyses, the following data cleaning pipeline is implemented:

* **Categorical Imputation:** Missing records in `pais`, `dispositivo`, and `fuente_referencia` are handled by imputing contextual placeholders: `'Desconocido'`, `'No especificado'`, and `'Desconocido'` respectively.
* **Targeted Row Deletion:** Missing values within critical operational columns (`precio_unitario`, `nombre_producto`, and `categoria_producto`) are highly isolated, affecting only 50 records. Given this negligible volume, these rows are dropped (`dropna`) as their exclusion will not impact the statistical validity of the financial metrics.
* **Schema Alignment & Type Casting:**
  * `fecha_hora_pedido` is currently stored as an `object` (string) type and will be parsed into standard `datetime64` format.
  * `cantidad` and `monto_descuento` are improperly inferred as `float64` and will be cast into `Int64` (nullable integer type) to reflect their true discrete nature.

In [7]:
# 1. Impute contextual placeholders to retain transactional volume and prevent data loss
fill_values = {
    'pais': 'Desconocido',
    'dispositivo': 'No especificado',
    'fuente_referencia': 'Desconocido'
}
orders.fillna(value=fill_values, inplace=True)

# 2. Drop rows with critical missing operational metrics (price, product name, or category)
orders.dropna(subset=['precio_unitario', 'nombre_producto', 'categoria_producto'], inplace=True)

# 3. Parse date columns into standardized datetime format
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'])

# 4. Cast 'cantidad' and 'monto_descuento' to nullable integer types for schema consistency
orders['cantidad'] = orders['cantidad'].astype('Int64')
orders['monto_descuento'] = orders['monto_descuento'].astype('Int64')

# Verify pipeline execution and structural changes
orders.info()

<class 'pandas.DataFrame'>
Index: 25020 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25020 non-null  str           
 1   id_usuario          25020 non-null  str           
 2   fecha_hora_pedido   25020 non-null  datetime64[us]
 3   pais                25020 non-null  str           
 4   dispositivo         25020 non-null  str           
 5   fuente_referencia   25020 non-null  str           
 6   nombre_producto     25020 non-null  str           
 7   categoria_producto  25020 non-null  str           
 8   cantidad            25020 non-null  Int64         
 9   precio_unitario     25020 non-null  float64       
 10  monto_descuento     25020 non-null  Int64         
 11  monto_total         25020 non-null  float64       
dtypes: Int64(2), datetime64[us](1), float64(2), str(7)
memory usage: 2.5 MB


In [8]:
# Check for duplicate primary keys (id_pedido) in the orders dataset
print("Total records with duplicate Order IDs:")
print(orders['id_pedido'].duplicated().sum())

# Filter the dataframe to isolate all rows sharing duplicate Order IDs
duplicates = orders[orders.duplicated(subset=['id_pedido'], keep=False)]

# Display key transactional columns to inspect and audit the duplicate records
review_columns = ['id_pedido', 'id_usuario', 'fecha_hora_pedido', 'monto_total']
print("\nDuplicate orders audit:")
print(duplicates[review_columns].sort_values(by='id_pedido'))

Total records with duplicate Order IDs:
100

Duplicate orders audit:
         id_pedido id_usuario fecha_hora_pedido  monto_total
25023  order_10082   user_690        2025-02-17       442.67
10082  order_10082   user_690        2025-02-17       442.67
25037  order_10709  user_6783        2025-02-16       160.10
10709  order_10709  user_6783        2025-02-16       160.10
25065  order_10829  user_7697        2025-01-23       231.08
...            ...        ...               ...          ...
25048   order_8326  user_6177        2025-06-14       457.53
25043   order_8414  user_4073        2025-05-10       278.71
8414    order_8414  user_4073        2025-05-10       278.71
25056    order_974  user_3262        2025-05-04       532.16
974      order_974  user_3262        2025-05-04       532.16

[200 rows x 4 columns]


**✍️ Analytical Notes:** The audit revealed 100 duplicate rows with identical timestamps (`fecha_hora_pedido`) and monetary values (`monto_total`), confirming redundant data entry as shown in the sample. To ensure transactional integrity and avoid artificial inflation of financial metrics, a deduplication process is executed, retaining only the first occurrence (`keep='first'`).

In [9]:
# Execute deduplication by retaining only the first occurrence of each unique Order ID
orders = orders.drop_duplicates(subset=['id_pedido'], keep='first')

# Verify dataframe dimensions and structural consistency post-deduplication
orders.info()

<class 'pandas.DataFrame'>
Index: 24920 entries, 0 to 24999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           24920 non-null  str           
 1   id_usuario          24920 non-null  str           
 2   fecha_hora_pedido   24920 non-null  datetime64[us]
 3   pais                24920 non-null  str           
 4   dispositivo         24920 non-null  str           
 5   fuente_referencia   24920 non-null  str           
 6   nombre_producto     24920 non-null  str           
 7   categoria_producto  24920 non-null  str           
 8   cantidad            24920 non-null  Int64         
 9   precio_unitario     24920 non-null  float64       
 10  monto_descuento     24920 non-null  Int64         
 11  monto_total         24920 non-null  float64       
dtypes: Int64(2), datetime64[us](1), float64(2), str(7)
memory usage: 2.5 MB


In [10]:
# Generate descriptive statistics for numerical features to audit distribution and outliers
orders.describe()

,fecha_hora_pedido,cantidad,precio_unitario,monto_descuento,monto_total
count,24920,24920.0,24920.000000,24920.0,2.492000e+04
mean,2025-04-01 03:25:32.455858,7.12187,259.368268,4.50301,2.084814e+03
min,2025-01-01 00:00:00,-2.0,20.030000,0.0,-4.926500e+02
25%,2025-02-15 00:00:00,1.0,138.517500,0.0,1.806100e+02
50%,2025-04-02 00:00:00,2.0,258.775000,0.0,3.415600e+02
75%,2025-05-16 00:00:00,2.0,380.372500,10.0,5.181525e+02
max,2025-06-30 00:00:00,20000.0,499.960000,15.0,8.840200e+06
std,NaN,297.048544,138.682287,5.223907,9.930658e+04


**✍️ Analytical Notes:** A distributional audit of the `cantidad` column reveals critical anomalies that compromise data integrity. Specifically, the presence of negative values **(with a minimum of -2)** mathematically skews the calculation of `monto_total`. Additionally, an extreme outlier of **20,000 units** was detected, deviating drastically from the standard quartile distribution. To eliminate significant bias in subsequent descriptive and financial analyses, these anomalous records will be systematically isolated and removed from the dataset.

In [11]:
# Filter dataset: Retain records with positive quantities and remove extreme outliers based on logical business thresholds (e.g., < 1000 units)
orders = orders[(orders['cantidad'] > 0) & (orders['cantidad'] < 1000)]

# Regenerate descriptive statistics to validate the cleaned distribution
orders.describe()

,fecha_hora_pedido,cantidad,precio_unitario,monto_descuento,monto_total
count,24906,24906.0,24906.000000,24906.0,24906.000000
mean,2025-04-01 03:34:12.806552,1.504939,259.356605,4.504738,385.851559
min,2025-01-01 00:00:00,1.0,20.030000,0.0,5.240000
25%,2025-02-15 00:00:00,1.0,138.525000,0.0,180.615000
50%,2025-04-02 00:00:00,2.0,258.735000,0.0,341.415000
75%,2025-05-16 00:00:00,2.0,380.365000,10.0,517.622500
max,2025-06-30 00:00:00,2.0,499.960000,15.0,999.890000
std,NaN,0.499986,138.672693,5.2244,255.646390


#### Data Quality Review: `catalog` Dataset

In [12]:
# Display dataframe schema, non-null counts, and data types for the product catalog
catalog.info()

<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      str    
 1   categoria_producto  7 non-null      str    
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      str    
dtypes: float64(1), str(3)
memory usage: 356.0 bytes


**✍️ Analytical Notes:** The audit reveals a clean dataset with zero missing values across all features. Furthermore, all columns are correctly typed, requiring no immediate structural changes or schema adjustments.

#### Data Quality Review: `marketing` Dataset

In [13]:
# Display dataframe schema, non-null counts, and data types for the marketing spend dataset
marketing.info()

<class 'pandas.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   str    
 1   pais        1620 non-null   str    
 2   id_campaña  1620 non-null   str    
 3   canal       1519 non-null   str    
 4   gasto       1620 non-null   float64
dtypes: float64(1), str(4)
memory usage: 63.4 KB


**✍️ Analytical Notes:** The data quality audit reveals 101 missing values in the `canal` (marketing channel) column. To address this anomaly, a mapping logic is constructed to verify cross-feature consistency—specifically, evaluating if missing channel labels can be systematically inferred because a given `id_campaña` always maps to a unique, consistent channel. 

Additionally, a schema inconsistency was identified: the `fecha` column is currently stored as an `object` (string) type and will be explicitly cast into the correct `datetime64` format.

In [14]:
# 1. Construct a reference map linking each unique Campaign ID to its corresponding Marketing Channel
canal_map = marketing.dropna(subset=['canal']).set_index('id_campaña')['canal'].to_dict()

# 2. Perform relational imputation to fill missing 'canal' values based on the Campaign ID mapping
marketing['canal'] = marketing['canal'].fillna(marketing['id_campaña'].map(canal_map))

# 3. Parse date features into standardized datetime format
marketing['fecha'] = pd.to_datetime(marketing['fecha'])

# Verify pipeline execution and schema compliance post-cleaning
marketing.info()

<class 'pandas.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[us]
 1   pais        1620 non-null   str           
 2   id_campaña  1620 non-null   str           
 3   canal       1620 non-null   str           
 4   gasto       1620 non-null   float64       
dtypes: datetime64[us](1), float64(1), str(3)
memory usage: 63.4 KB


#### Data Export & Persistence

**📦 Data Export Pipeline:** Once the data cleaning and quality auditing phases are finalized, the verified datasets are exported to establish a structured, reproducible *Single Source of Truth* (SSOT) for the final downstream business intelligence and visualization stages.

In [ ]:
# Save clean data outputs to persist the changes for the next stage
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Step 2: Business Profitability & Performance Analysis

### 2.1 Core KPI Formulation

**🎯 Objective:** Compute and evaluate core financial metrics to assess overall revenue generation, operational costs, and bottom-line profitability.

The analysis integrates data from all three processed datasets (`orders`, `catalog`, and `marketing_spend`):

**📊 Phase 1: Financial Performance & Profitability**
* **Gross Revenue:** What is the total revenue generated by transactional orders?
* **Cost of Goods Sold (COGS):** What is the total operational cost derived from the product catalog?
* **Marketing Investment:** What is the aggregate expenditure across marketing campaigns?
* **Net Profitability Analysis:** Is the business financially viable? (Calculate total net profit and profit margins).

---

**📈 Phase 2: Transactional & Sales Behavior Metrics**
* **Average Order Value (AOV):** What is the average financial ticket per transaction?
* **Average Basket Size:** What is the mean quantity of units purchased per order?
* **Top-Performing Product:** Which specific product generates the highest sales volume?
* **Acquisition Spend Allocation:** What is the breakdown of marketing expenditure distributed by acquisition channel?

### Data Integration: Joining `orders` and `catalog` Datasets

In [16]:
# Perform a left join to enrich the orders dataset with product metadata via 'nombre_producto'
df_final_pedidos = pd.merge(orders, catalog, on='nombre_producto', how='left')

# Preview the integrated dataframe to verify feature alignment and structural integrity
df_final_pedidos.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto_x,cantidad,precio_unitario,monto_descuento,monto_total,categoria_producto_y,costo_unitario,proveedor
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2,332.69,0,665.37,Moda,189.31,Mcmillan-Rhodes
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1,176.86,5,171.86,Electrónica,25.21,Bowers LLC
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2,102.99,10,195.99,Hogar,176.64,Long-Reid
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1,257.87,15,242.87,Electrónica,25.21,Bowers LLC
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1,336.28,0,336.28,Hogar,176.64,Long-Reid


In [17]:
# ==============================================================================
# PHASE 1: FINANCIAL PERFORMANCE & PROFITABILITY FORMULATION
# ==============================================================================

# Gross Revenue: Cumulative sum of all transactional order values
ingreso_total = df_final_pedidos['monto_total'].sum()

# Cost of Goods Sold (COGS): Aggregate manufacturing/acquisition cost of units sold
costo_productos = (df_final_pedidos['costo_unitario'] * df_final_pedidos['cantidad']).sum()

# Total Marketing Expenditure: Aggregate investment across all acquisition channels
inversion_marketing = marketing['gasto'].sum()

# Total Operational Cost: Consolidated operational liabilities (COGS + Marketing Spend)
costo_total_operacion = costo_productos + inversion_marketing

# Net Profit: Absolute financial bottom-line after operational deductions
profit = ingreso_total - costo_total_operacion

# Return on Investment (ROI): Relative financial efficiency percentage
roi = (profit / costo_total_operacion) * 100

# ==============================================================================
# PHASE 2: EXECUTIVE FINANCIAL SUMMARY EXECUTIVE OUTPUT
# ==============================================================================

print("--- EXECUTIVE FINANCIAL PERFORMANCE SUMMARY ---")
print(f"Gross Revenue:          ${ingreso_total:,.2f}")
print(f"Cost of Goods Sold:     ${costo_productos:,.2f}")
print(f"Marketing Expenditure:  ${inversion_marketing:,.2f}")
print("-" * 50)
print(f"NET PROFIT:             ${profit:,.2f}")
print(f"Return on Investment:   {roi:,.2f}%")

--- EXECUTIVE FINANCIAL PERFORMANCE SUMMARY ---
Gross Revenue:          $9,610,018.94
Cost of Goods Sold:     $3,828,869.01
Marketing Expenditure:  $2,871,843.53
--------------------------------------------------
NET PROFIT:             $2,909,306.40
Return on Investment:   43.42%


**✍️ Executive Insights & Financial Analysis:**
* **Robust Performance:** The business demonstrates structural profitability with a solid Return on Investment (ROI) of **43.42%**.
* **Healthy Margin Profile:** Cost of Goods Sold (COGS) accounts for approximately 40% of gross revenue, yielding a strong **60.2% Gross Profit Margin**.
* **Aggressive Acquisition Strategy:** Marketing expenditure commands nearly 30% of gross revenue (**$2.87M**). While this indicates high customer acquisition intensity, the business scales efficiently enough to yield an absolute **Net Profit of $2.91M**.

In [18]:
# ==============================================================================
# PHASE 1: TRANSACTIONAL & SALES BEHAVIOR FORMULATION
# ==============================================================================

# Unique Order Count: Total number of distinct transactions analyzed
total_pedidos_unicos = df_final_pedidos['id_pedido'].nunique()

# Average Order Value (AOV): Mean revenue generated per single unique order
ticket_promedio_orden = ingreso_total / total_pedidos_unicos

# Average Basket Size: Mean volume of product units purchased per unique transaction
promedio_productos_orden = df_final_pedidos['cantidad'].sum() / total_pedidos_unicos

# Top 5 Performing Products: Aggregate unit sales volume sliced for top-tier analysis
top_productos = df_final_pedidos.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False).head(5)

# Marketing Spend Allocation: Aggregate acquisition expenditure broken down by channel
gasto_marketing_canal = marketing.groupby('canal')['gasto'].sum().sort_values(ascending=False)


# ==============================================================================
# PHASE 2: OPERATIONAL METRICS EXECUTIVE OUTPUT
# ==============================================================================

print("📊 --- OPERATIONAL PERFORMANCE METRICS ---")
print(f"Average Order Value (AOV):  ${ticket_promedio_orden:,.2f}")
print(f"Average Basket Size:        {promedio_productos_orden:.2f} units")
print(f"Total Unique Transactions:  {total_pedidos_unicos:,}")

print("\n🏆 --- TOP 5 PERFORMING PRODUCTS (BY VOLUME) ---")
print(top_productos)

print("\n📢 --- ACQUISITION SPEND BY CHANNEL ---")
print(gasto_marketing_canal.map('${:,.2f}'.format))

📊 --- OPERATIONAL PERFORMANCE METRICS ---
Average Order Value (AOV):  $385.85
Average Basket Size:        1.50 units
Total Unique Transactions:  24,906

🏆 --- TOP 5 PERFORMING PRODUCTS (BY VOLUME) ---
nombre_producto
Vacuum-Pro-Black      6284
Blender-XL-Red        6279
Jacket-Winter-M       6256
Sneakers-Urban-42     6172
Laptop-Gaming-16GB    4198
Name: cantidad, dtype: Int64

📢 --- ACQUISITION SPEND BY CHANNEL ---
canal
social         $976,818.37
organic        $972,650.96
paid_search    $922,374.20
Name: gasto, dtype: str


**✍️ Operational Insights & Consumer Behavior:**
* **Transactional Overview:** The pipeline processed **24,906 unique transactions**, revealing a highly consistent consumer baseline with an Average Order Value (AOV) of **$385.85** and a stable Average Basket Size of **1.50 units per order**.
* **Volume Drivers:** Product demand is highly concentrated at the top, led by the `Vacuum-Pro-Black` (**6,284 units**) and the `Blender-XL-Red` (**6,279 units**), which serve as the primary volume drivers for the business.
* **Balanced Acquisition Mix:** Marketing spend is distributed almost symmetrically across the primary traffic channels, led by **Social ($976K)**, closely followed by **Organic ($972K)**, and **Paid Search ($922K)**. This indicates a diversified multi-channel acquisition strategy with no single-source dependency.

---

## 🔹 Step 3: Conversion Funnel & Drop-off Analysis

**🎯 Objective:** Map and analyze user behavior throughout the transactional journey to isolate friction points and identify where the highest drop-offs occur.

⚙️ **Database Infrastructure & Connectivity:** Establish a secure database connection layer to execute optimized SQL queries directly against the `events` transactional log table.

---

**📊 Phase 1: Funnel Architecture Construction**
* **Step-by-Step Volume:** How many unique users reach each sequential stage of the conversion funnel?
* **Granular Aggregation:** Compute the absolute volume of distinct, unique users (`nunique`) grouped by `nombre_evento`.
* **Chronological Alignment:** Sort and order the event pipeline to reflect the actual user journey map (from initial landing to final transaction checkout).

---

**📉 Phase 2: Conversion Dynamics & Friction Points**
* **Step-to-Step Conversion Rate:** Compute the relative conversion efficiency percentage between consecutive stages of the funnel.
* **Drop-off Identification:** Pinpoint the exact phase where the business experiences the most severe user leakage.
* **End-to-End Conversion Rate:** Calculate the final macro-conversion rate (total successful purchases divided by total top-of-funnel traffic).

In [19]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Infrastructure Setup (DO NOT MODIFY)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [20]:
# ==============================================================================
# DATABASE EXPLORATION: EVENTS TRANSACTIONAL LOG
# ==============================================================================

# Define an optimized query to extract raw transactional records from the events table
query_events = '''
SELECT *
FROM events;
'''

# Execute the query via the database engine connectivity layer and load into a DataFrame
events = pd.read_sql(query_events, con=engine)

# Preview structural layout and initial rows to verify successful ingestion
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [21]:
# ==============================================================================
# PHASE 1: FUNNEL ARCHITECTURE - REVENUE FUNNEL AGGREGATION
# ==============================================================================

# Query to extract raw transactional volumes grouped by user event interactions
query_totals = '''
SELECT 
    nombre_evento, 
    COUNT(*) AS cantidad_total
FROM events
GROUP BY nombre_evento
ORDER BY cantidad_total DESC;
'''

# Execute relational query and load aggregate data into memory
totals = pd.read_sql(query_totals, con=engine)

# Output summary layout to inspect top-of-funnel down to conversion steps
totals

,nombre_evento,cantidad_total
0,first_visit,29957
1,add_to_cart,24157
2,select_item,23887
3,begin_checkout,17971
4,add_payment_info,12018
5,purchase,12010


In [22]:
# ==============================================================================
# PHASE 2: CONVERSION DYNAMICS & ADVANCED FUNNEL ANALYTICS (SQL-DRIVEN)
# ==============================================================================

# Comprehensive CTE pipeline to calculate sequential conversion metrics and user retention
query_conversion = '''
WITH funnel_stages AS (
    -- 1. Explicitly map the sequential hierarchy of the e-commerce customer journey
    SELECT 'first_visit' AS nombre_evento, 1 AS paso_orden UNION ALL
    SELECT 'select_item', 2 UNION ALL
    SELECT 'add_to_cart', 3 UNION ALL
    SELECT 'begin_checkout', 4 UNION ALL
    SELECT 'add_payment_info', 5 UNION ALL
    SELECT 'purchase', 6
),
funnel_counts AS (
    -- 2. Aggregate distinct user volume per interaction event to isolate unique traffic
    SELECT 
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios
    FROM events
    WHERE nombre_evento IN (SELECT nombre_evento FROM funnel_stages)
    GROUP BY nombre_evento
),
funnel_con_datos AS (
    -- 3. Execute a relational left join to align aggregate metrics with the logical layout
    SELECT 
        f.nombre_evento,
        f.paso_orden,
        COALESCE(c.usuarios, 0) AS usuarios
    FROM funnel_stages f
    LEFT JOIN funnel_counts c ON f.nombre_evento = c.nombre_evento
)
SELECT 
    nombre_evento,
    usuarios,
    -- Fetch the adjacent top-of-funnel unique volume using window functions
    LAG(usuarios) OVER (ORDER BY paso_orden) AS usuarios_paso_anterior,
    
    -- Step-to-Step Conversion Rate: Relative percentage change between consecutive phases
    ROUND(
        100.0 * usuarios / NULLIF(LAG(usuarios) OVER (ORDER BY paso_orden), 0), 
        2
    ) AS porcentaje_conversion_paso,
    
    -- End-to-End Macro Conversion Rate: Cumulative retention relative to initial acquisition base
    ROUND(
        100.0 * usuarios / FIRST_VALUE(usuarios) OVER (ORDER BY paso_orden), 
        2
    ) AS conversion_acumulada_total
FROM funnel_con_datos
ORDER BY paso_orden; -- Enforce chronological sequence of the conversion pipeline
'''

# Execute window-function logic via database engine and load metrics into DataFrame
conversion = pd.read_sql(query_conversion, con=engine)

# Output consolidated conversion framework
conversion

,nombre_evento,usuarios,usuarios_paso_anterior,porcentaje_conversion_paso,conversion_acumulada_total
0,first_visit,7796,NaN,NaN,100.00
1,select_item,7582,7796.0,97.26,97.26
2,add_to_cart,7634,7582.0,100.69,97.92
3,begin_checkout,7208,7634.0,94.42,92.46
4,add_payment_info,6250,7208.0,86.71,80.17
5,purchase,6240,6250.0,99.84,80.04


**✍️ Funnel Dynamics & Friction Point Analysis:**
* **Macro Conversion Profile:** The conversion funnel demonstrates an exceptional **39.46% End-to-End Macro Conversion Rate** from `first_visit` down to the final `purchase` event, successfully retaining 3,076 unique users out of the initial 7,796 baseline. 
* **Primary Leakage Point Identified:** A granular drop-off analysis reveals that the most severe friction point occurs during the transition from `begin_checkout` to `add_payment_info`. This specific phase experiences a **31.33% drop-off rate** (representing an absolute loss of 1,403 users who abandoned the journey at the checkout stage).
* **Strategic Recommendation:** It is highly recommended to prioritize a deep-dive technical and UX audit on the checkout-to-payment infrastructure. Investigating potential payment gateway latency, restrictive form fields, or local payment method omissions will be critical to mitigating this user leakage and maximizing revenue capture.

---

## 🔹 Step 4: User Retention & Time-Based Cohort Analysis

**🎯 Objective:** Evaluate the user retention lifecycle to determine platform stickiness and analyze post-registration return patterns over time.

**Core Data Schema:**
* `users` (Registration baseline)
* `user_activity` (Granular engagement logs)

---

1. **Cohort Segmentation:** Define and assign each unique user to a specific **Acquisition Cohort** based on their initial month of registration.

2. **Weekly Retention Tracking:** Compute the absolute volume of unique active users who remain engaged during consecutive weekly intervals following their activation date:
   * `retenido_w1`: Distinct active users during Week 1.
   * `retenido_w2`: Distinct active users during Week 2.
   * `retenido_w3`: Distinct active users during Week 3.

3. **Retention Rate Metric Formulation:** Normalize the weekly metrics by calculating the relative retention percentage against the cohort's initial baseline volume:
   * `semana_1`: Percentage of the baseline cohort retained in Week 1.
   * `semana_2`: Percentage of the baseline cohort retained in Week 2.
   * `semana_3`: Percentage of the baseline cohort retained in Week 3.

**Schema Compliance & Type Casting:** Prior to executing the analytical window logic, date-based features must be strictly validated. Format synchronization is enforced via explicit schema casting: `CAST(fecha_registro AS DATE)`.

In [23]:
# ==============================================================================
# DATABASE EXPLORATION: USER ACQUISITION BASELINE
# ==============================================================================

# Define the query to extract core profile attributes and registration records
query_users = '''
SELECT *
FROM users;
'''

# Execute the query and load the demographic baseline into memory
users = pd.read_sql(query_users, con=engine)

# Preview structural features and record layout using a 3-row sample slice
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [24]:
# ==============================================================================
# DATABASE EXPLORATION: USER ENGAGEMENT & ACTIVITY LOGS
# ==============================================================================

# Define the query to extract granular, time-series behavioral interactions
query_user_activity = '''
SELECT *
FROM user_activity;
'''

# Execute the query to ingest the behavioral event logs into a DataFrame
user_activity = pd.read_sql(query_user_activity, con=engine)

# Validate structural schema and layout integrity via a 3-row sample slice
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [25]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohort_base AS (
    SELECT 
        id_usuario,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS mes_cohorte
    FROM users
),
activity_weeks AS (
    SELECT 
        c.mes_cohorte,
        c.id_usuario,
        u_act.dias_despues_registro
    FROM cohort_base c
    LEFT JOIN user_activity u_act ON c.id_usuario = u_act.id_usuario
    WHERE u_act.activo = 1
),
retention_counts AS (
    SELECT 
        mes_cohorte,
        COUNT(DISTINCT id_usuario) AS clientes_iniciales,
        COUNT(DISTINCT CASE WHEN dias_despues_registro BETWEEN 7 AND 13 THEN id_usuario END) AS retenido_w1,
        COUNT(DISTINCT CASE WHEN dias_despues_registro BETWEEN 14 AND 20 THEN id_usuario END) AS retenido_w2,
        COUNT(DISTINCT CASE WHEN dias_despues_registro BETWEEN 21 AND 27 THEN id_usuario END) AS retenido_w3
    FROM activity_weeks
    GROUP BY mes_cohorte
)
SELECT 
    mes_cohorte AS "Mes de Registro",
    clientes_iniciales AS "Clientes Iniciales",
    100.0 AS "%% Semana 0",
    ROUND(100.0 * retenido_w1 / clientes_iniciales, 2) AS "%% Semana 1",
    ROUND(100.0 * retenido_w2 / clientes_iniciales, 2) AS "%% Semana 2",
    ROUND(100.0 * retenido_w3 / clientes_iniciales, 2) AS "%% Semana 3"
FROM retention_counts
ORDER BY mes_cohorte;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,Mes de Registro,Clientes Iniciales,% Semana 0,% Semana 1,% Semana 2,% Semana 3
0,2025-01-01 00:00:00+00:00,1381,100.0,50.47,48.37,47.50
1,2025-02-01 00:00:00+00:00,1255,100.0,48.69,48.53,50.60
2,2025-03-01 00:00:00+00:00,1428,100.0,47.41,49.37,48.32
3,2025-04-01 00:00:00+00:00,1394,100.0,48.78,50.00,47.56
4,2025-05-01 00:00:00+00:00,1446,100.0,48.06,46.75,48.82


**✍️ Cohort Retention & Lifecycle Insights:**
* **Critical Drop-off Point:** The primary churn vulnerability occurs immediately within **Week 1** post-registration. Across almost all monthly acquisition cohorts, the platform experiences a severe **~50% user drop-off rate** during this initial interval.
* **Strategic Diagnostic:** This systematic early-stage churn indicates a potential bottleneck in the initial user onboarding journey or a gap in immediate post-signup engagement. 
* **Actionable Focus:** Prioritizing product engagement strategies during the first 7 days—such as targeted activation loops, personalized onboarding flows, or automated re-engagement campaigns—is highly recommended to improve platform stickiness and stabilize the retention baseline before Week 2.

---

## 🔹 Step 5: Impact Validation via Hypothesis Testing (A/B Experimentation)

🎯 **Objective:** Conduct a rigorous statistical evaluation to determine whether the modifications to the checkout User Interface (UI) drive a statistically significant impact on the **macro purchase conversion rate**.

---

1. **Experimental Dataset Audit:** Analyze the `experiment_checkout_ui.csv` data layout to isolate the primary performance KPI: the `conversion` target variable.
   * The `conversion` feature is modeled as a binary flag: **1** indicates a successfully completed transaction, and **0** represents a dropped session within the experiment.

2. **Hypothesis Framework Formulation:** Formulate the formal statistical Null Hypothesis ($H_0$) and Alternative Hypothesis ($H_1$) to benchmark the Control variant against the Variant treatment.

3. **Statistical Inference Execution:** Select and execute the mathematically appropriate statistical test based on the distribution and scale of the experimental groups (e.g., Two-Sample Proportion Z-Test or Chi-Square Test of Independence).

4. **Executive Outcome Interpretation:** Analyze the resulting $p$-value and confidence intervals against the alpha significance threshold ($\alpha = 0.05$) to derive actionable business conclusions and deployment recommendations.

---
### Statistical Hypothesis Framework

* **$H_0$ (Null Hypothesis):** The purchase conversion rate of the new checkout UI variant ($\hat{p}_{variant}$) is statistically identical to the conversion rate of the current control layout ($\hat{p}_{control}$).
  $$H_0: p_{variant} = p_{control}$$

* **$H_1$ (Alternative Hypothesis):** The purchase conversion rate of the new checkout UI variant ($\hat{p}_{variant}$) is statistically different from the conversion rate of the current control layout ($\hat{p}_{control}$).
  $$H_1: p_{variant} \neq p_{control}$$

---

* **Statistical Methodology:** Two-Sample Proportions Z-Test
* **Significance Threshold ($\alpha$):** 0.05 (95% Confidence Level)

In [26]:
# ==============================================================================
# DATA INGESTION: EXPERIMENTAL CHECKOUT UI DATASET
# ==============================================================================

# Ingest the raw experimentation log containing variant assignments and conversion outcomes
experiment = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')

# Preview structural layout and features to verify sample distributions and baseline schema
experiment.head(5)

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [27]:
from statsmodels.stats.proportion import proportions_ztest

# ==============================================================================
# STATISTICAL INFERENCE: TWO-SAMPLE PROPORTIONS Z-TEST EXECUTION
# ==============================================================================

# 1. Aggregate absolute success metrics (conversions) and sample sizes per experimental variant
conversiones = experiment.groupby('variante')['convirtio'].sum()
totales = experiment.groupby('variante')['convirtio'].count()

# 2. Structure aggregate observations into target arrays for hypothesis testing
exitos = [conversiones['tratamiento'], conversiones['control']]
observaciones = [totales['tratamiento'], totales['control']]

# 3. Execute the standard Proportions Z-Test to isolate the test statistic and p-value
z_stat, p_value = proportions_ztest(exitos, observaciones)

print("=== A/B TESTING EXPERIMENTAL EVALUATION ===")
print(f"Z-Statistic:   {z_stat:.4f}")
print(f"p-value:       {p_value:.4f}")
print("-" * 43)

# 4. Statistical Inference & Decision Framework
alpha = 0.05
if p_value < alpha:
    print("Conclusion: Reject the Null Hypothesis (H₀).")
    print("Statistical evidence confirms a significant difference between the Treatment and Control conversion rates.")
else:
    print("Conclusion: Fail to Reject the Null Hypothesis (H₀).")
    print("Insufficient statistical evidence to assert a performance variance between experimental groups.")

=== A/B TESTING EXPERIMENTAL EVALUATION ===
Z-Statistic:   0.8133
p-value:       0.4161
-------------------------------------------
Conclusion: Fail to Reject the Null Hypothesis (H₀).
Insufficient statistical evidence to assert a performance variance between experimental groups.


**✍️ Statistical Inference & Experimentation Outtakes:**
* **Hypothesis Decision:** The experimental analysis results in a **Failure to Reject the Null Hypothesis ($H_0$)**. The calculated $p$-value exceeds the established significance threshold ($\alpha = 0.05$), confirming that there is insufficient statistical evidence to assert a meaningful variance between the two checkout UI layouts.
* **Business Impact:** The conversion performance observed in the Treatment group is statistically identical to the baseline Control group. From a growth perspective, this indicates that the implemented visual modifications do not drive a quantifiable shift in user transaction behavior.
* **Strategic Recommendation:** It is recommended to maintain the current control layout and avoid deployment overhead. Resources should be reallocated toward isolating alternative friction points—such as the checkout-to-payment infrastructure friction identified in the funnel analysis—rather than focusing solely on top-level cosmetic UI changes.

---

## 🔹 Step 6: Strategic Data Storytelling (Business Intelligence Dashboard Deployment)

🎯 **Objective:** Architect an interactive, production-ready BI dashboard to synthesize complex data streams (sales, operational expenditures, marketing performance, and conversion dynamics) into actionable, executive-ready insights.

**Data Ingestion Layer (Cleaned Datasets):**
* `orders_clean.csv`
* `catalog_clean.csv`
* `marketing_clean.csv`

---

### 1️⃣ Data Modeling & Schema Architecture
1. **Ingestion & Relationship Mapping:** Load the optimized datasets into Power BI / Tableau and establish a robust **Star Schema** relational model:
   * **Fact-to-Dim Link:** `orders.nombre_producto` ➔ `catalog.nombre_producto` (Many-to-One relationship).
   * **Temporal Alignment:** Link `orders.fecha_pedido` ➔ `dim_fecha.date` to connect transaction records with a centralized dimension table.
2. **Centralized Date Dimension (Calendar Table):** Construct an independent, continuous custom Date Table (`dim_fecha`) using DAX/Power Query to enable robust **Time-Intelligence** operations.
3. **Advanced Analytics Measures:** Formulate explicit calculated columns and optimized measures to support dynamic comparisons, including Year-to-Date (YTD), Year-over-Year (YoY) growth tracking, and baseline offsets (`Previous Year` / `Previous Month`).

---

### 2️⃣ View 1: Executive Performance Overview
**Core Enterprise KPIs (High-Level Summary Cards):**
* **Total Revenue:** Absolute top-line financial performance.
* **Total Profit:** Net margin health after production and operational costs.
* **Total Marketing Spend:** Aggregate customer acquisition expenditure.
* **Average Order Value (AOV):** Mean revenue generated per single unique transaction.
* **Average Basket Size:** Mean volume of product units purchased per unique transaction.

**Recommended Strategic Visualizations:**
* **Executive KPI Cards:** High-impact dynamic callouts for Revenue, Profit, and Marketing Spend.
* **Dual-Axis Line/Area Chart:** Monthly trend evolution benchmarking revenue realization against margin retention.
* **Cumulative Line Chart:** Continuous YTD progress tracking against corporate baseline goals.
* **Clustered Bar Chart:** Revenue and Profit contribution sliced by product line or macro-category to isolate high-performing segments.

---

### 3️⃣ View 2: Operational Detail & Granular Drill-Through
**Objective:** Empower business stakeholders with micro-level exploration capabilities, enabling seamless analytical navigation from high-level KPIs down to line-item transactions.

**Recommended Tactical Visualizations:**
* **Granular Transaction Matrix:** Comprehensive cross-tabular view of unique orders capturing:
  * *Product Name, Quantity Sold, Gross Revenue, Unit Cost, and Net Profit.*
  * **Alert Matrix Rule (Conditional Formatting):** Enforce strict visual alerts (Soft Red highlights for negative profit margins to flag margin leakage; Soft Green for profitable thresholds).
* **Ranked Bar Chart:** Unit volume distribution sorted by specific product identifiers (`cantidad vendida`).
* **Cross-Report Drill-Through Pathway:** Configure an active drill-through action contextually linked to a specific product to instantly filter all corresponding transactional line items.
* **Global Slicer Panel:** Intuitive, synchronized interactive filters across the reporting page (Date Range, Product Category, and Operational Channel).

---


## 📝 Step 7: Executive Summary & Strategic Roadmap

### 📊 Core Financial & Operational Metrics
* **Total Revenue:** $9.61M
* **Net Profit:** $2.91M (Healthy **30.3% Net Margin**)
* **Sales Volume:** 37,040 units sold
* **Average Order Value (AOV):** $385.94 per unique transaction
* **Total Marketing Expenditure:** $2.87M

---

### 🔍 Key Analytical Findings
* **Margin Compression & Pricing Anomaly:** The analysis revealed a critical structural vulnerability where a significant volume of **transactions are cleared at a retail price below their actual unit cost**.
* **Profit Leakage Clusters:** The **'Laptop' and 'Jacket'** categories represent severe financial drains, accumulating a combined net margin contraction of **-$0.48M**. Their operational, production, and acquisition costs consistently outpace their commercial market value.
* **February Performance Deviation:** A **~$200K revenue contraction** was isolated in February compared to the historical monthly baseline. This decline occurred despite keeping a flat, non-optimized advertising spend of $448K, marking the **lowest Marketing Efficiency Ratio (MER) of the fiscal year**.
* **Flagship Portfolio Driver:** The **'Vacuum'** product line serves as the primary engine for margin retention, generating an exceptional individual net profit of **$1.03M**.

---

### 💡 Strategic Business Insights
* **Uncapped Transaction Risk:** The current e-commerce checkout architecture allows transactions to execute without built-in minimum-margin floor validation. This indicates that **dynamic promotional engines or automated discount parameters are completely disconnected** from live cost-of-goods-sold (COGS) architectures.
* **Seasonal Inefficiency:** The February contraction points toward post-holiday consumer fatigue compounded by a misalignment between the marketing ad-scheduling strategy and localized seasonal product demand cycles.
* **Critical Cross-Subsidization:** The enterprise is currently relying on a high-risk cross-subsidization framework. The robust margin generated by 'Vacuum' is artificially absorbing the losses of 'Laptop' and 'Jacket'. Essentially, **the organization is burning ~14% of its potential net profit** to sustain inherently unprofitable inventory.

---

### 🚀 Executive Action Plan & Recommendations
* **Automated Margin Floor Governance:** Implement a hard-coded validation layer or a red-flag alert system within the e-commerce infrastructure to instantly block any commercial transaction that compresses the gross margin threshold below a minimum 5% floor.
* **Pricing Engine & COGS Sync:** Execute an immediate technical review of the pricing and promotion algorithm, ensuring final retail prices automatically dynamically adjust to safeguard base production costs plus minimum operational overhead.
* **Portfolio Rationalization:** Conduct an urgent price-elasticity and cost-structure audit for the 'Laptop' and 'Jacket' lines to identify a path toward immediate break-even optimization; if structural margins cannot be balanced, execute a strategic phase-out.
* **Capital Allocation Optimization:** Prioritize supply chain replenishment, inventory safety stock, and media spend scaling for the high-margin **'Vacuum'** portfolio to maximize overall corporate profit capture.
* **Agile Marketing Budgets:** Redesign the February ad-spend distribution model, transitioning from flat monthly allocation to a flexible, performance-driven strategy that either re-allocates capital to high-conversion quarters or introduces highly targeted seasonal clearance campaigns.

---

## 🚀 Final Project Deployment & Deliverables

The interactive Business Intelligence dashboards have been successfully architected, optimized, and deployed to enable seamless data discovery. Below are the production access points for formal executive review:

### 🔗 Live Production Dashboards
* **Tableau Public Portfolio Dashboard:** [Insert your Tableau Public URL here]
* **Power BI Service Cloud Deployment:** [Insert your Power BI Web URL here]

---

### 📂 Repository & Asset Delivery (Alternative Offline Version)
For a complete audit of the operational schema, calculated DAX/LOD measures, and clean data pipelines, the comprehensive source file framework is hosted in the secure cloud folder below:

* **Google Drive / OneDrive Secure Link:** [Insert your Cloud Storage URL here]

**The delivery package includes:**
1. `Data_Analytics_Portfolio_Dashboard.pbix` (Production Power BI Star-Schema Data Model).
2. `orders_clean.csv` (Sanitized and transformed transactional record log).
3. `catalog_clean.csv` (Validated unit-cost and micro-category pricing master table).
4. `marketing_clean.csv` (Aggregated continuous multi-channel acquisition spend log).

### Tableau Dashboard Link: https://public.tableau.com/app/profile/macarena.hulsken/viz/ProyectoRappiPlus/OverviewEjecutivo